# 청년 복지정보 크롤러 (중앙정부 카테고리)

## 개요
Selenium과 BeautifulSoup을 사용해 정부 복지정보 포털의 **중앙정부** 카테고리에서 청년 대상 복지 서비스 정보를 자동으로 수집하는 크롤러입니다. 1~11페이지에 게시된 전체 항목을 순회하며 각 항목의 상세 페이지에 진입해 지원대상, 서비스내용, 신청방법, 추가정보 등 세부 필드를 추출한 뒤 CSV / TSV / JSON 세 가지 형식으로 저장합니다.

> **저작권 안내**: 이 코드가 접근하는 실제 사이트 도메인은 저작권 문제로 `target_site`라는 이름으로 치환하였습니다.

## 주요 기능
1. **라벨 매핑 (`LABEL_MAPPING`)** — 사이트마다 다르게 표기되는 항목명(예: "서비스 내용" vs "지원 내용")을 하나의 표준 필드명으로 통일합니다.
2. **동적 콘텐츠 대기 (`wait_for_dynamic_content`)** — JS로 렌더링되는 요소가 로드될 때까지 대기한 뒤 파싱을 시작합니다.
3. **탭별 파싱 함수**
   - `extract_process_steps` — 처리절차 섹션 추출 (Selenium 우선, 실패 시 BeautifulSoup으로 재시도)
   - `extract_detail_content_standard` — 지원대상 / 서비스내용 / 신청방법 탭 파싱
   - `extract_additional_info_content` — 추가정보 탭(문의처, 관련웹사이트, 근거법령 등) 파싱, 정규식으로 전화번호·URL 보조 추출
4. **`extract_detail_info`** — 상세 페이지 진입 후 제목, 담당부처, 표 정보, 탭 데이터를 모두 모아 하나의 딕셔너리로 구성합니다.
5. **메인 루프** — 페이지(1~11) → 항목("자세히 보기" 버튼) 순서로 순회하며 `상세 페이지 방문 → 데이터 수집 → 목록으로 복귀`를 반복합니다.
6. **`save_data_to_files`** — 수집된 데이터를 pandas `DataFrame`으로 변환한 뒤 CSV / TSV / JSON 세 형식으로 저장합니다.

## 코드 구조
- **1. 유틸리티 및 매핑 설정** — 라벨 매핑, 텍스트 정제(`clean_text`), 탭 XPath 정의
- **2. 크롤링 로직** — 탭별 상세 정보 추출 함수들
- **3. 메인 실행 블록** — Selenium 드라이버 설정, 페이지/항목 반복, 데이터 저장

## 설계 포인트
- **다중 fallback 파싱**: 각 필드마다 'Selenium 요소 탐색 → 실패 시 BeautifulSoup 텍스트 매칭 → 정규식 보조 추출' 순으로 최대 2~3단계까지 시도하여 사이트 구조 변경이나 요소 누락에 대응합니다.
- **중복 텍스트 제거**: `seen_contents` 집합으로 페이지 인트로 텍스트가 여러 필드에 중복 삽입되는 것을 방지합니다.
- **차단 회피 설정**: headless Chrome + User-Agent 위장 등으로 자동화 탐지를 우회하는 옵션을 포함합니다.


In [ ]:
## 타겟사이트 청년 복지정보 크롤러 - 중앙정부 1~11페이지 전체 항목
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from bs4 import BeautifulSoup
import pandas as pd
import time
import sys
import os
import json
import re

# ========================================
# 1. 유틸리티 및 매핑 설정
# ========================================

# 모든 섹션 레이블 정의 및 매핑
LABEL_MAPPING = {
    '지원대상': '지원대상', '서비스 내용': '서비스내용', '지원 내용': '서비스내용',
    '신청방법': '신청방법', '추가정보': '추가정보', '선정기준': '선정기준',
    '신청기간': '신청기간', '접수기관': '접수기관', '문의처': '문의처',
    '전화문의': '문의처', '관련 웹사이트': '관련웹사이트', '근거법령': '근거법령',
    '서식/자료': '서식자료', '복지사례 알아보기': '복지사례', '처리절차': '처리절차'
}

# 탭 ID와 구분을 매핑
TAB_SECTIONS = {1: '중앙정부', 2: '지자체', 3: '민간'}

def clean_text(text):
    """텍스트 정리: 불필요한 공백/개행 제거 및 잡음 필터링"""
    if text is None: return ""
    text = text.replace("이 누리집은 대한민국 공식 전자정부 누리집입니다.", "")
    text = text.replace("임신 및 출산에 장애가 될 수 있는 건강위험요인의 조기 발견 기회를 제공하고, 임신전 건강관리를 위한 의료.보건학적 지원을 통해 건강한 임신 출산 환경을 조성합니다.", "")
    text = text.replace("[새창열림]링크 이동", "")
    return ' '.join(text.split()).strip()

# 탭 텍스트 기반 XPath
TAB_INFO = [
    ('지원대상', "//div[normalize-space(text())='지원대상']"),
    ('서비스 내용', "//div[normalize-space(text())='서비스 내용']"),
    ('신청방법', "//div[normalize-space(text())='신청방법']"),
    ('추가정보', "//div[normalize-space(text())='추가정보']"),
]

# ========================================
# 2. 개선된 크롤링 로직
# ========================================

def wait_for_dynamic_content(driver, timeout=5):
    """동적 콘텐츠 로딩 대기 (여러 방법 시도)"""
    try:
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.cl-htmlsnippet, div.wlfare-step-tit"))
        )
    except TimeoutException:
        pass

    time.sleep(2)
    driver.execute_script("return document.readyState")
    time.sleep(1)


def extract_process_steps(driver, debug=False):
    """처리절차 정보 추출 - 개선 버전"""
    process_data = {}
    try:
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # 방법 1: 원래 방식
        process_containers = driver.find_elements(By.CSS_SELECTOR, "div.wlfare-step-tit")
        if process_containers:
            steps = []
            for container in process_containers:
                try:
                    step_title = clean_text(container.text)
                    parent = container.find_element(By.XPATH, "./../..")
                    try:
                        step_content_element = parent.find_element(By.CSS_SELECTOR, "div.wlfare-step-cont div.cl-text")
                        step_content = clean_text(step_content_element.text)
                        if step_title and step_content:
                            steps.append(f"{step_title}: {step_content}")
                    except NoSuchElementException:
                        if step_title: steps.append(step_title)
                except Exception:
                    continue
            if steps:
                process_data['처리절차'] = '\n'.join(steps)

        # 방법 2: BeautifulSoup으로 재시도
        if not process_data.get('처리절차'):
            process_sections = soup.find_all('div', class_='wlfare-step-tit')
            if process_sections:
                steps = []
                for section in process_sections:
                    title = clean_text(section.get_text())
                    if title:
                        parent = section.find_parent('div', class_='wlfare-step-cont')
                        if parent:
                            content = clean_text(parent.get_text().replace(title, ''))
                            if content:
                                steps.append(f"{title}: {content}")
                        else:
                            steps.append(title)
                if steps:
                    process_data['처리절차'] = '\n'.join(steps)

    except Exception as e:
        if debug:
            print(f"    ⚠️ 처리절차 추출 오류: {e}")

    return process_data


def extract_detail_content_standard(driver, page_intro_text="", debug=False):
    """일반 탭(지원대상, 서비스내용, 신청방법)의 내용 추출 - 개선 버전"""
    tab_data = {}
    seen_contents = set()

    if page_intro_text:
        seen_contents.add(page_intro_text)

    wait_for_dynamic_content(driver)

    soup = BeautifulSoup(driver.page_source, 'html.parser')

    if not page_intro_text:
        try:
            intro_elements = driver.find_elements(By.CSS_SELECTOR, "div.cl-htmlsnippet")
            if intro_elements:
                page_intro_text = clean_text(intro_elements[0].text)
                if page_intro_text:
                    seen_contents.add(page_intro_text)
        except:
            pass

    try:
        active_pane = driver.find_element(By.CSS_SELECTOR, "div.cl-layout-content[data-role='content-pane']")
        containers = active_pane.find_elements(By.CSS_SELECTOR, "div.cl-layout-wrap")
    except:
        containers = driver.find_elements(By.CSS_SELECTOR, "div.cl-layout-wrap")

    for idx, container in enumerate(containers):
        try:
            section_title = ''
            try:
                section_title_element = container.find_element(By.CSS_SELECTOR, "div.line-tit div.cl-text")
                section_title_text = clean_text(section_title_element.text)

                if section_title_text == '처리절차' or section_title_text not in LABEL_MAPPING:
                    continue
                section_title = LABEL_MAPPING[section_title_text]
            except NoSuchElementException:
                continue

            content = ''
            try:
                content_element = container.find_element(By.CSS_SELECTOR, "div.cl-htmlsnippet")

                list_items = content_element.find_elements(By.CSS_SELECTOR, "ul.bokjiContsIn li")

                if list_items:
                    li_texts = []
                    for li in list_items:
                        text = clean_text(li.text)

                        if text and text != page_intro_text and text not in seen_contents:
                            if page_intro_text and page_intro_text in text:
                                text = text.replace(page_intro_text, '').strip()

                            if text:
                                li_texts.append(text)
                                seen_contents.add(text)
                    content = '\n'.join(li_texts)
                else:
                    content = clean_text(content_element.text)

                    if page_intro_text:
                        if page_intro_text == content:
                            content = ''
                        elif page_intro_text in content:
                            content = content.replace(page_intro_text, '').strip()

                    if content and content not in seen_contents:
                        seen_contents.add(content)
                    else:
                        content = ''

                if not content:
                    all_text_elements = content_element.find_elements(By.CSS_SELECTOR, "div, p, span, li")
                    texts = []
                    for elem in all_text_elements:
                        text = clean_text(elem.text)

                        if text and text != page_intro_text and text not in seen_contents:
                            if page_intro_text and page_intro_text in text:
                                text = text.replace(page_intro_text, '').strip()

                            if text:
                                texts.append(text)
                                seen_contents.add(text)
                    content = '\n'.join(texts)

                if content:
                    if section_title in tab_data:
                        tab_data[section_title] += '\n\n' + content
                    else:
                        tab_data[section_title] = content
            except NoSuchElementException:
                pass
        except Exception as e:
            if debug:
                print(f"    ⚠️ 컨테이너 처리 오류: {e}")

    for label_text, mapped_key in LABEL_MAPPING.items():
        if mapped_key not in tab_data or not tab_data[mapped_key]:
            label_elements = soup.find_all(string=re.compile(f'^{re.escape(label_text)}$'))
            for label_elem in label_elements:
                parent = label_elem.find_parent(['div', 'h1', 'h2', 'h3', 'h4'])
                if parent:
                    content_container = parent.find_next_sibling()
                    if content_container:
                        content = clean_text(content_container.get_text())

                        if page_intro_text and page_intro_text in content:
                            content = content.replace(page_intro_text, '').strip()

                        if content and len(content) > 5 and content not in seen_contents and content != page_intro_text:
                            seen_contents.add(content)
                            if mapped_key in tab_data:
                                tab_data[mapped_key] += '\n\n' + content
                            else:
                                tab_data[mapped_key] = content
                            break

    process_data = extract_process_steps(driver, debug=debug)
    tab_data.update(process_data)

    return tab_data


def extract_additional_info_content(driver, page_intro_text="", debug=False):
    """추가정보 탭의 내용 추출 - 개선 버전"""
    current_tab_data = {}

    seen_contents = set()
    if page_intro_text:
        seen_contents.add(page_intro_text)

    wait_for_dynamic_content(driver)

    soup = BeautifulSoup(driver.page_source, 'html.parser')

    try:
        content_area = driver.find_element(By.CSS_SELECTOR, "div.cl-layout-content[data-role='content-pane']")
        containers = content_area.find_elements(By.XPATH, ".//div[contains(@class, 'line-tit')]")

        for section_title_element in containers:
            section_title = ''
            section_title_text = ''
            try:
                text_element = section_title_element.find_element(By.CSS_SELECTOR, "div.cl-text")
                section_title_text = clean_text(text_element.text)
                if section_title_text not in LABEL_MAPPING:
                    continue
                section_title = LABEL_MAPPING[section_title_text]
            except NoSuchElementException:
                continue

            try:
                content_root_element = section_title_element.find_element(By.XPATH, "./../../../../..")
                raw_text = content_root_element.text
                content = clean_text(raw_text.replace(section_title_text, "").strip())

                if page_intro_text and page_intro_text in content:
                    content = content.replace(page_intro_text, '').strip()

                if content and content not in seen_contents:
                    seen_contents.add(content)
                    final_key = section_title
                    if final_key in current_tab_data:
                        current_tab_data[final_key] += '\n\n' + content
                    else:
                        current_tab_data[final_key] = content
            except NoSuchElementException:
                pass
    except NoSuchElementException:
        pass
    except Exception as e:
        if debug:
            print(f"    ⚠️ 추가정보 추출 오류: {e}")

    for label_text, mapped_key in LABEL_MAPPING.items():
        if mapped_key not in current_tab_data or not current_tab_data[mapped_key]:
            if mapped_key in ['문의처', '관련웹사이트', '근거법령', '서식자료']:
                label_elements = soup.find_all(string=re.compile(f'{re.escape(label_text)}'))
                for label_elem in label_elements:
                    parent_container = label_elem.find_parent(['div', 'section', 'article'])
                    if parent_container:
                        content = clean_text(parent_container.get_text().replace(label_text, ''))

                        if page_intro_text and page_intro_text in content:
                            content = content.replace(page_intro_text, '').strip()

                        if content and len(content) > 5 and content not in seen_contents:
                            seen_contents.add(content)
                            if mapped_key in current_tab_data:
                                current_tab_data[mapped_key] += '\n\n' + content
                            else:
                                current_tab_data[mapped_key] = content
                            break

    page_text = driver.page_source

    if '문의처' not in current_tab_data or not current_tab_data['문의처']:
        phone_pattern = r'(\d{2,3}[-\s]?\d{3,4}[-\s]?\d{4})'
        phones = re.findall(phone_pattern, soup.get_text())
        if phones:
            current_tab_data['문의처'] = ', '.join(set(phones[:3]))

    if '관련웹사이트' not in current_tab_data or not current_tab_data['관련웹사이트']:
        url_pattern = r'https?://[^\s<>"\']+(?:[^\s<>"\'\)])'
        urls = re.findall(url_pattern, page_text)
        if urls:
            external_urls = [url for url in urls if 'target_site.go.kr' not in url]
            if external_urls:
                current_tab_data['관련웹사이트'] = '\n'.join(set(external_urls[:5]))

    return current_tab_data


def extract_detail_info(driver, current_tab_id, debug=False):
    """상세 페이지에서 탭을 순서대로 클릭하여 모든 정보 추출 - 개선 버전"""
    data = {
        '상세URL': driver.current_url,
        '제목': '제목 없음',
        '구분': TAB_SECTIONS.get(current_tab_id, '알 수 없음')
    }

    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.wlfare-info-nm div.cl-text"))
        )
        time.sleep(4)

        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # 제목 추출
        try:
            title_element = driver.find_element(By.CSS_SELECTOR, "div.wlfare-info-nm div.cl-text")
            data['제목'] = clean_text(title_element.text)
        except:
            pass

        # 담당부처 정보 추출
        try:
            dept_xpath = "//div[contains(@aria-label, '담당부처') or contains(text(), '담당부처')]/../following-sibling::div//div[contains(@class, 'cl-text')]"
            try:
                dept_element = driver.find_element(By.XPATH, dept_xpath)
                data['담당부처'] = clean_text(dept_element.text)
            except:
                pass

            if '담당부처' not in data or not data['담당부처']:
                try:
                    dept_label = driver.find_element(By.CSS_SELECTOR, "div.diamond-blt")
                    if '담당부처' in dept_label.text:
                        parent = dept_label.find_element(By.XPATH, "../..")
                        dept_value = parent.find_element(By.XPATH, ".//div[@aria-label='복지정보담당부처명']")
                        data['담당부처'] = clean_text(dept_value.text)
                except:
                    pass

            if '담당부처' not in data or not data['담당부처']:
                dept_labels = soup.find_all(string=re.compile('담당부처'))
                for label in dept_labels:
                    parent = label.find_parent('div')
                    if parent:
                        next_elem = parent.find_next_sibling()
                        if next_elem:
                            dept_text = clean_text(next_elem.get_text())
                            if dept_text and len(dept_text) > 2:
                                data['담당부처'] = dept_text
                                break

                        grandparent = parent.find_parent('div')
                        if grandparent:
                            for elem in grandparent.find_all('div'):
                                if elem != parent and '담당부처' not in elem.get_text():
                                    text = clean_text(elem.get_text())
                                    if text and len(text) > 3 and '담당부처' not in text:
                                        data['담당부처'] = text
                                        break
                        if '담당부처' in data:
                            break

        except Exception as e:
            if debug:
                print(f"         담당부처 추출 실패: {e}")

        # 서비스 세부 내용 추출 (탭 클릭 전)
        page_intro_text = ""
        try:
            intro_elements = driver.find_elements(By.CSS_SELECTOR, "div.cl-htmlsnippet")
            if intro_elements:
                page_intro_text = clean_text(intro_elements[0].text)

                if page_intro_text and len(page_intro_text) > 10:
                    data['서비스세부내용'] = page_intro_text

        except Exception as e:
            if debug:
                print(f"         서비스 세부 내용 추출 실패: {e}")

        # 표 정보 추출
        try:
            table = driver.find_element(By.CSS_SELECTOR, "table")

            headers = table.find_elements(By.CSS_SELECTOR, "thead th")
            values = table.find_elements(By.CSS_SELECTOR, "tbody td")

            if len(headers) == len(values):
                for header, value in zip(headers, values):
                    header_text = clean_text(header.text)
                    value_text = clean_text(value.text)

                    if header_text and value_text:
                        data[f'표_{header_text}'] = value_text

            table_soup = soup.find('table')
            if table_soup:
                headers_soup = table_soup.find_all('th')
                values_soup = table_soup.find_all('td')

                if len(headers_soup) == len(values_soup):
                    for h, v in zip(headers_soup, values_soup):
                        h_text = clean_text(h.get_text())
                        v_text = clean_text(v.get_text())

                        if h_text and v_text:
                            data[f'표_{h_text}'] = v_text

        except Exception as e:
            if debug:
                print(f"         표 정보 추출 실패: {e}")

        all_tab_data = {}

        # 각 탭 순회
        for tab_name, tab_xpath in TAB_INFO:
            try:
                if debug:
                    print(f"      → '{tab_name}' 탭 처리 중...")

                tab_element = WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.XPATH, tab_xpath))
                )

                try:
                    clickable_element = tab_element.find_element(By.XPATH, "./../..")
                except NoSuchElementException:
                    clickable_element = tab_element

                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", clickable_element)
                time.sleep(0.5)

                driver.execute_script("arguments[0].click();", clickable_element)

                time.sleep(3)

                wait_for_dynamic_content(driver, timeout=5)

                if tab_name == '추가정보':
                    current_tab_data = extract_additional_info_content(driver, page_intro_text, debug=debug)
                else:
                    current_tab_data = extract_detail_content_standard(driver, page_intro_text, debug=debug)

                if debug and current_tab_data:
                    print(f"         ✓ {len(current_tab_data)}개 필드 추출: {list(current_tab_data.keys())}")

                all_tab_data.update(current_tab_data)

            except Exception as e:
                if debug:
                    print(f"         ✗ '{tab_name}' 탭 처리 실패: {e}")
                pass

        data.update(all_tab_data)

    except Exception as e:
        if debug:
            print(f"      ✗ 상세 정보 추출 실패: {e}")
        pass

    return data

# ========================================
# 3. 메인 실행 블록
# ========================================

def save_data_to_files(data_list, filename_prefix, columns_order):
    """주어진 데이터를 DataFrame으로 만들어 CSV, TSV, JSON 파일로 저장"""
    if not data_list:
        print(f"⚠️ 저장할 데이터가 없습니다: {filename_prefix}")
        return []

    df = pd.DataFrame(data_list)
    existing_columns = [col for col in columns_order if col in df.columns]
    df = df.reindex(columns=existing_columns, fill_value='')

    saved_files = []

    # 1. TSV 저장
    tsv_filename = f'{filename_prefix}.tsv'
    df.to_csv(tsv_filename, index=False, sep='\t', encoding='utf-8-sig')
    print(f"\n🎉 {len(data_list)}개 항목을 '{tsv_filename}' (TSV)에 저장했습니다.")
    saved_files.append(tsv_filename)

    # 2. CSV 저장
    csv_filename = f'{filename_prefix}.csv'
    df.to_csv(csv_filename, index=False, sep=',', encoding='utf-8-sig')
    print(f"🎉 {len(data_list)}개 항목을 '{csv_filename}' (CSV)에 저장했습니다.")
    saved_files.append(csv_filename)

    # 3. JSON 저장
    json_filename = f'{filename_prefix}.json'
    df.to_json(json_filename, orient='records', force_ascii=False, indent=4)
    print(f"🎉 {len(data_list)}개 항목을 '{json_filename}' (JSON)에 저장했습니다.")
    saved_files.append(json_filename)

    return saved_files

print("✅ 타겟겟사이트 청년 복지정보 통합 크롤링을 시작합니다. (중앙정부 1~11페이지 전체 항목)")
print("="*70)

# --- 설정 변수 ---
DEBUG_MODE = False  # 디버그 모드 (True로 설정하면 상세 로그 출력)
COLLECT_ALL_ITEMS = True  # ⭐ True: 모든 항목 수집, False: 페이지당 9개만 수집

# 1. base_url
base_url = "https://www.target_site.go.kr/ssis-tbu/twataa/wlfareInfo/moveTWAT52005M.do"

# 2. 크롤링할 탭 ID (1: 중앙정부)
TAB_IDS_TO_CRAWL = [1]

# 3. 크롤링할 페이지 범위
START_PAGE = 1
END_PAGE = 11

# 파일 저장에 필요한 컬럼 순서
COLUMNS_ORDER = ['순번', '페이지', '구분', '제목', '서비스세부내용', '담당부처',
                 '표_기준연도', '표_문의처', '표_지원주기', '표_제공유형',
                 '지원대상', '선정기준', '서비스내용', '신청방법',
                 '처리절차', '신청기간', '접수기관', '문의처', '관련웹사이트', '근거법령',
                 '서식자료', '복지사례', '추가정보', '상세URL']

# 드라이버 설정
chrome_options = Options()
if not DEBUG_MODE:
    chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--lang=ko-KR')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_experimental_option('prefs', {'intl.accept_languages': 'ko,ko-KR'})
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

try:
    driver = webdriver.Chrome(options=chrome_options)
    driver.maximize_window()
    driver.execute_cdp_cmd('Network.setUserAgentOverride', {
        "userAgent": 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        "acceptLanguage": "ko-KR,ko;q=0.9"
    })
except Exception as e:
    print(f"❌ 드라이버 초기화 오류: {e}")
    sys.exit()

all_section_files = []

# --- 크롤링 루프 시작 ---
try:
    for tab_id in TAB_IDS_TO_CRAWL:
        section_name = TAB_SECTIONS.get(tab_id, f'tabId={tab_id}')

        section_data = []
        file_prefix = f'복지사이트_청년복지_{section_name}_{START_PAGE}~{END_PAGE}페이지_전체항목'

        print(f"\n\n{'#'*70}")
        print(f"🎯 섹션 크롤링 시작: {section_name} (tabId={tab_id})")
        print(f"   📊 크롤링 범위: {START_PAGE}~{END_PAGE}페이지")
        print(f"   📈 수집 모드: {'전체 항목' if COLLECT_ALL_ITEMS else '페이지당 9개'}")
        print(f"{'#'*70}")

        # 페이지 루프
        for page in range(START_PAGE, END_PAGE + 1):
            print(f"\n{'='*60}")
            print(f"📖 {section_name} - {page}/{END_PAGE} 페이지 크롤링 중...")
            print(f"{'='*60}")

            url = f"{base_url}?page={page}&orderBy=date&tabId={tab_id}&period=%EC%B2%AD%EB%85%84"
            driver.get(url)
            time.sleep(5)

            button_xpath = "//a[contains(@aria-label, '자세히 보기') or contains(text(), '자세히 보기')]"

            try:
                WebDriverWait(driver, 15).until(
                    EC.presence_of_element_located((By.XPATH, button_xpath))
                )
                buttons = driver.find_elements(By.XPATH, button_xpath)
            except:
                print(f"\n❌ '{section_name}' - '{page} 페이지'에서 버튼을 찾을 수 없습니다. 다음 페이지로 이동합니다.")
                continue

            # ⭐ 모든 항목 또는 9개만 처리
            if COLLECT_ALL_ITEMS:
                items_to_process = len(buttons)
            else:
                items_to_process = min(9, len(buttons))

            print(f"\n총 {len(buttons)}개 항목 중 {items_to_process}개 처리...")

            for i in range(items_to_process):
                try:
                    # 매번 버튼 목록 새로 가져오기 (DOM 변경 대비)
                    current_buttons = driver.find_elements(By.XPATH, button_xpath)

                    if i >= len(current_buttons):
                        break

                    button_to_click = current_buttons[i]
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", button_to_click)
                    time.sleep(0.5)

                    print(f"  ➡️ [{i+1}/{items_to_process}] 항목 클릭...")

                    driver.execute_script("arguments[0].click();", button_to_click)
                    time.sleep(3)

                    # 상세 정보 추출
                    detail_data = extract_detail_info(driver, tab_id, debug=DEBUG_MODE)

                    if detail_data and detail_data.get('상세URL'):
                        detail_data['페이지'] = page
                        detail_data['순번'] = len(section_data) + 1
                        section_data.append(detail_data)

                        # 수집된 필드 카운트
                        filled_fields = sum(1 for v in detail_data.values() if v and v != '제목 없음')
                        total_fields = len(COLUMNS_ORDER)

                        print(f"     ✅ {detail_data['제목'][:40]}")
                        print(f"        수집 필드: {filled_fields}/{total_fields}개")

                        # 누락된 주요 필드 표시
                        important_fields = ['지원대상', '서비스내용', '신청방법', '문의처']
                        missing = [f for f in important_fields if not detail_data.get(f)]
                        if missing:
                            print(f"        ⚠️ 누락: {', '.join(missing)}")
                    else:
                        print(f"     ❌ 정보 추출 실패")

                    # 목록 페이지로 돌아가기
                    driver.back()
                    time.sleep(3)

                except Exception as e:
                    print(f"     ⚠️ 오류 ({i+1}번째): {type(e).__name__}")
                    try:
                        driver.get(url)
                        time.sleep(5)
                    except:
                        break
                    continue

            # 페이지별 진행상황 출력
            print(f"\n📊 현재까지 수집된 항목: {len(section_data)}개")

        # 섹션별 저장
        if section_data:
            print(f"\n\n--- {section_name} 섹션 저장 시작 ---")
            print(f"📦 총 수집 항목: {len(section_data)}개")
            saved_files = save_data_to_files(section_data, file_prefix, COLUMNS_ORDER)
            all_section_files.extend(saved_files)
            print(f"--- {section_name} 저장 완료 ---\n")
        else:
            print(f"\n⚠️ {section_name} 섹션에서 수집된 데이터가 없습니다.")

except Exception as e:
    print(f"\n❌ 크롤링 중 오류: {e}")
    if DEBUG_MODE:
        import traceback
        traceback.print_exc()

finally:
    try:
        driver.quit()
    except:
        pass

print("\n" + "="*70)
print("🎉 크롤링 완료!")
print(f"📊 총 {len(section_data) if 'section_data' in locals() else 0}개 항목 수집")
print(f"📁 저장된 파일: {len(all_section_files)}개")
for file in all_section_files:
    print(f"   - {file}")
print("="*70)